# Notebook 19 — Eval-Driven Rollouts, Regressions, and Incident Playbooks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/19_eval_driven_rollouts_regressions_and_incident_playbooks.ipynb)

Reliable systems do not promote a candidate build because it feels promising. They promote it because eval results, operational telemetry, and rollback plans all agree that the risk is acceptable. This notebook turns Module 5 observability into release discipline.

## What you will build

- a release-comparison table grounded in `evals/`-style quality metrics
- rollout gates that combine quality, latency, reliability, and cost budgets
- a canary simulation that shows how regressions surface over time
- explicit rollback design artifacts
- incident playbooks and a rehearsal report for a realistic runtime regression

## Reconnecting to `evals/`

Notebook 18 measured what the runtime is doing. This notebook decides what to do **about** those measurements. The same logic from `evals/` still applies:

- compare candidate releases against a known-good baseline
- define thresholds before the rollout starts
- classify regressions as blockers, warnings, or acceptable drift
- record artifacts that make rollback and incident response faster than ad-hoc judgment

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add notebook helpers and artifact paths

We will export every major output: release comparisons, canary windows, rollback plans, and incident playbooks.

In [ ]:
random.seed(19)

ARTIFACT_DIR = ARTIFACT_ROOT / "notebook_19_rollouts_and_incidents"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def clipped(value, low=0.0, high=1.0):
    return max(low, min(high, value))

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define baseline and candidate release summaries

These suites intentionally look like the earlier `evals/` track: core quality, schema adherence, tool behavior, robustness, and ops telemetry. The goal is to make rollout review feel like a continuation of evaluation, not a separate discipline.

In [ ]:
release_results = [
    {"release": "stable_2026_05_01", "suite": "quality_core", "rubric_score": 0.88, "schema_pass_rate": 0.98, "tool_success_rate": 0.95, "robustness_pass_rate": 0.93, "p95_latency_ms": 1180, "error_rate": 0.014, "cost_per_1k_req": 6.8},
    {"release": "stable_2026_05_01", "suite": "long_context", "rubric_score": 0.85, "schema_pass_rate": 0.97, "tool_success_rate": 0.93, "robustness_pass_rate": 0.90, "p95_latency_ms": 1355, "error_rate": 0.018, "cost_per_1k_req": 7.5},
    {"release": "stable_2026_05_01", "suite": "structured_outputs", "rubric_score": 0.87, "schema_pass_rate": 0.995, "tool_success_rate": 0.96, "robustness_pass_rate": 0.92, "p95_latency_ms": 1260, "error_rate": 0.012, "cost_per_1k_req": 7.2},
    {"release": "stable_2026_05_01", "suite": "robustness_adversarial", "rubric_score": 0.82, "schema_pass_rate": 0.97, "tool_success_rate": 0.91, "robustness_pass_rate": 0.95, "p95_latency_ms": 1295, "error_rate": 0.016, "cost_per_1k_req": 7.1},
    {"release": "candidate_2026_05_18", "suite": "quality_core", "rubric_score": 0.89, "schema_pass_rate": 0.982, "tool_success_rate": 0.956, "robustness_pass_rate": 0.934, "p95_latency_ms": 1165, "error_rate": 0.013, "cost_per_1k_req": 6.6},
    {"release": "candidate_2026_05_18", "suite": "long_context", "rubric_score": 0.86, "schema_pass_rate": 0.969, "tool_success_rate": 0.928, "robustness_pass_rate": 0.904, "p95_latency_ms": 1470, "error_rate": 0.024, "cost_per_1k_req": 7.9},
    {"release": "candidate_2026_05_18", "suite": "structured_outputs", "rubric_score": 0.88, "schema_pass_rate": 0.973, "tool_success_rate": 0.954, "robustness_pass_rate": 0.921, "p95_latency_ms": 1315, "error_rate": 0.019, "cost_per_1k_req": 7.4},
    {"release": "candidate_2026_05_18", "suite": "robustness_adversarial", "rubric_score": 0.83, "schema_pass_rate": 0.971, "tool_success_rate": 0.914, "robustness_pass_rate": 0.948, "p95_latency_ms": 1342, "error_rate": 0.017, "cost_per_1k_req": 7.3},
]

release_df = pd.DataFrame(release_results)
release_df.to_csv(ARTIFACT_DIR / "release_eval_summary.csv", index=False)
print(release_df.to_markdown(index=False))

## Step 3: Define rollout gates before reading the candidate result

Good rollout policy is pre-committed policy. If you invent thresholds after a regression appears, you are doing storytelling instead of release engineering.

In [ ]:
gate_config = {
    "rubric_score_delta": -0.01,
    "schema_pass_rate_delta": -0.005,
    "tool_success_rate_delta": -0.01,
    "robustness_pass_rate_delta": -0.01,
    "p95_latency_ms_delta": 120,
    "error_rate_delta": 0.007,
    "cost_per_1k_req_delta": 0.6,
    "absolute": {
        "schema_pass_rate_min": 0.975,
        "robustness_pass_rate_min": 0.90,
        "p95_latency_ms_max": 1450,
        "error_rate_max": 0.025,
    },
}

baseline_df = release_df[release_df["release"] == "stable_2026_05_01"].set_index("suite")
candidate_df = release_df[release_df["release"] == "candidate_2026_05_18"].set_index("suite")
print(json.dumps(gate_config, indent=2))

## Step 4: Compare candidate vs baseline and classify regressions

We will compute deltas by suite, then classify each suite as pass, warning, or blocker. This mirrors the release regression logic from `evals/`, but now latency, reliability, and cost sit beside quality metrics.

In [ ]:
comparison_rows = []
for suite in baseline_df.index:
    base = baseline_df.loc[suite]
    cand = candidate_df.loc[suite]
    delta_row = {
        "suite": suite,
        "rubric_score_delta": round(float(cand["rubric_score"] - base["rubric_score"]), 4),
        "schema_pass_rate_delta": round(float(cand["schema_pass_rate"] - base["schema_pass_rate"]), 4),
        "tool_success_rate_delta": round(float(cand["tool_success_rate"] - base["tool_success_rate"]), 4),
        "robustness_pass_rate_delta": round(float(cand["robustness_pass_rate"] - base["robustness_pass_rate"]), 4),
        "p95_latency_ms_delta": round(float(cand["p95_latency_ms"] - base["p95_latency_ms"]), 2),
        "error_rate_delta": round(float(cand["error_rate"] - base["error_rate"]), 4),
        "cost_per_1k_req_delta": round(float(cand["cost_per_1k_req"] - base["cost_per_1k_req"]), 3),
    }
    blocker_rules = [
        delta_row["rubric_score_delta"] < gate_config["rubric_score_delta"],
        delta_row["schema_pass_rate_delta"] < gate_config["schema_pass_rate_delta"],
        delta_row["tool_success_rate_delta"] < gate_config["tool_success_rate_delta"],
        delta_row["robustness_pass_rate_delta"] < gate_config["robustness_pass_rate_delta"],
        delta_row["p95_latency_ms_delta"] > gate_config["p95_latency_ms_delta"],
        delta_row["error_rate_delta"] > gate_config["error_rate_delta"],
        delta_row["cost_per_1k_req_delta"] > gate_config["cost_per_1k_req_delta"],
        float(cand["schema_pass_rate"]) < gate_config["absolute"]["schema_pass_rate_min"],
        float(cand["robustness_pass_rate"]) < gate_config["absolute"]["robustness_pass_rate_min"],
        float(cand["p95_latency_ms"]) > gate_config["absolute"]["p95_latency_ms_max"],
        float(cand["error_rate"]) > gate_config["absolute"]["error_rate_max"],
    ]
    status = "blocker" if any(blocker_rules) else "pass"
    if status == "pass" and delta_row["p95_latency_ms_delta"] > 60:
        status = "warning"
    delta_row["status"] = status
    comparison_rows.append(delta_row)

comparison_df = pd.DataFrame(comparison_rows).sort_values(["status", "suite"])
comparison_df.to_csv(ARTIFACT_DIR / "release_comparison.csv", index=False)
print(comparison_df.to_markdown(index=False))

## Step 5: Simulate a canary rollout over time

Static comparisons are necessary but not sufficient. Some regressions only appear under live traffic shape, burstiness, or long-context load. We will simulate a staged canary with quality, schema, error, and latency signals.

In [ ]:
rollout_stages = [
    {"stage": "1%", "minutes": 15, "traffic_share": 0.01},
    {"stage": "10%", "minutes": 15, "traffic_share": 0.10},
    {"stage": "25%", "minutes": 15, "traffic_share": 0.25},
    {"stage": "50%", "minutes": 15, "traffic_share": 0.50},
]

canary_rows = []
minute_cursor = 0
for stage_index, stage in enumerate(rollout_stages):
    for minute in range(stage["minutes"]):
        rng = random.Random(f'stage-{stage["stage"]}-{minute}')
        latent_regression = 1 if stage["stage"] in {"25%", "50%"} and minute >= 6 else 0
        canary_rows.append({
            "minute": minute_cursor,
            "stage": stage["stage"],
            "traffic_share": stage["traffic_share"],
            "candidate_requests": int(1600 * stage["traffic_share"]),
            "rubric_score": round(0.886 - 0.01 * latent_regression + rng.uniform(-0.003, 0.003), 4),
            "schema_pass_rate": round(0.981 - 0.008 * latent_regression + rng.uniform(-0.002, 0.001), 4),
            "robustness_pass_rate": round(0.933 - 0.004 * latent_regression + rng.uniform(-0.002, 0.002), 4),
            "p95_latency_ms": round(1180 + 28 * stage_index + 145 * latent_regression + rng.uniform(-18, 18), 2),
            "error_rate": round(0.014 + 0.002 * stage_index + 0.008 * latent_regression + rng.uniform(-0.001, 0.001), 4),
            "cost_per_1k_req": round(6.6 + 0.12 * stage_index + 0.25 * latent_regression + rng.uniform(-0.05, 0.05), 3),
        })
        minute_cursor += 1

canary_df = pd.DataFrame(canary_rows)
canary_df.to_csv(ARTIFACT_DIR / "canary_windows.csv", index=False)
print(canary_df.head(12).to_markdown(index=False))

## Step 6: Turn canary windows into rollout decisions

The rollout controller should be boring: it reads metrics, applies policy, and returns continue, hold, or rollback.

In [ ]:
def evaluate_window(row):
    reasons = []
    if row["schema_pass_rate"] < gate_config["absolute"]["schema_pass_rate_min"]:
        reasons.append("schema")
    if row["robustness_pass_rate"] < gate_config["absolute"]["robustness_pass_rate_min"]:
        reasons.append("robustness")
    if row["p95_latency_ms"] > gate_config["absolute"]["p95_latency_ms_max"]:
        reasons.append("latency")
    if row["error_rate"] > gate_config["absolute"]["error_rate_max"]:
        reasons.append("errors")
    if row["cost_per_1k_req"] - 6.8 > gate_config["cost_per_1k_req_delta"]:
        reasons.append("cost")
    if row["rubric_score"] < 0.87:
        reasons.append("quality")
    if reasons:
        return "rollback", ",".join(sorted(set(reasons)))
    if row["p95_latency_ms"] > 1360 or row["error_rate"] > 0.02:
        return "hold", "observe"
    return "continue", "healthy"

decision_rows = []
for _, row in canary_df.iterrows():
    decision, reason = evaluate_window(row)
    decision_rows.append({**row.to_dict(), "decision": decision, "reason": reason})

decision_df = pd.DataFrame(decision_rows)
stage_decisions = (
    decision_df.groupby("stage")
    .agg(
        windows=("minute", "count"),
        rollback_windows=("decision", lambda values: int(sum(v == "rollback" for v in values))),
        hold_windows=("decision", lambda values: int(sum(v == "hold" for v in values))),
        worst_p95_latency_ms=("p95_latency_ms", "max"),
        worst_error_rate=("error_rate", "max"),
    )
    .reset_index()
)
stage_decisions["recommended_action"] = np.where(stage_decisions["rollback_windows"] > 0, "rollback", np.where(stage_decisions["hold_windows"] > 0, "hold", "promote"))
stage_decisions.to_csv(ARTIFACT_DIR / "stage_decisions.csv", index=False)
print(stage_decisions.to_markdown(index=False))

## Step 7: Design rollback before you need it

Rollback is not only “send traffic back.” You need a specific target build, config state, adapter mapping, cache policy, and validation checklist.

In [ ]:
rollback_plan = {
    "rollback_target": "stable_2026_05_01",
    "traffic_policy": [
        "set candidate traffic share to 0%",
        "pin all requests to stable runtime pool",
        "disable speculative decoding override introduced in candidate",
    ],
    "state_checkpoints": [
        "restore router manifest v41",
        "re-enable schema validator strict mode",
        "invalidate canary-only prompt cache namespace",
        "confirm fallback adapter map for castalia-mentor",
    ],
    "owners": {
        "runtime_oncall": "owns traffic switch",
        "eval_owner": "confirms post-rollback quality spot checks",
        "product_comms": "updates internal status page",
    },
    "exit_criteria": [
        "5 minutes of stable latency under baseline budget",
        "schema pass rate returns above 0.975",
        "error rate returns below 0.02",
    ],
}
write_json(ARTIFACT_DIR / "rollback_plan.json", rollback_plan)
print(json.dumps(rollback_plan, indent=2))

## Step 8: Build an incident taxonomy

Playbooks work best when incidents are classified consistently. We will define a small catalog that maps signals to likely owners and first-response questions.

In [ ]:
incident_catalog = [
    {"incident_id": "schema-regression", "severity": "SEV2", "trigger": "schema_pass_rate < 0.975", "owner": "runtime_oncall", "first_question": "Did the parser or validator config change?"},
    {"incident_id": "tail-latency-spike", "severity": "SEV2", "trigger": "p95_latency_ms > 1450", "owner": "serving_oncall", "first_question": "Is queueing or decode time dominating?"},
    {"incident_id": "error-burst", "severity": "SEV1", "trigger": "error_rate > 0.03", "owner": "platform_oncall", "first_question": "Is the runtime crashing or being throttled?"},
    {"incident_id": "cost-runaway", "severity": "SEV3", "trigger": "cost_per_1k_req > 7.4", "owner": "finops", "first_question": "Did cache hit rate collapse or did routing change?"},
]
incident_catalog_df = pd.DataFrame(incident_catalog)
incident_catalog_df.to_csv(ARTIFACT_DIR / "incident_catalog.csv", index=False)
print(incident_catalog_df.to_markdown(index=False))

## Step 9: Generate incident playbooks as concrete artifacts

Each playbook will include signals, immediate checks, rollback triggers, and communications guidance. These are the operational equivalents of eval rubrics: they standardize response quality.

In [ ]:
def build_playbook(record):
    return "\n".join([
        "# Incident Playbook: " + record["incident_id"],
        "",
        "## Trigger",
        "- " + record["trigger"],
        "",
        "## Owner",
        "- " + record["owner"],
        "",
        "## Immediate checks",
        "- Compare current canary window to stable baseline",
        "- Inspect trace breakdown from Notebook 18 artifacts",
        "- Verify whether the regression is route-specific or global",
        "- Confirm whether rollback criteria are already met",
        "",
        "## First question",
        "- " + record["first_question"],
        "",
        "## Rollback trigger",
        "- Trigger rollback immediately if two consecutive windows stay beyond policy threshold",
        "",
        "## Communications",
        "- Update internal incident channel",
        "- Log customer-facing status impact if blast radius exceeds one tenant",
    ])

playbook_rows = []
for record in incident_catalog:
    file_name = record["incident_id"] + "_playbook.md"
    playbook_path = ARTIFACT_DIR / file_name
    playbook_path.write_text(build_playbook(record), encoding="utf-8")
    playbook_rows.append({"incident_id": record["incident_id"], "playbook_file": file_name})

playbook_index = pd.DataFrame(playbook_rows)
playbook_index.to_csv(ARTIFACT_DIR / "playbook_index.csv", index=False)
print(playbook_index.to_markdown(index=False))

## Step 10: Rehearse a realistic regression incident

We will use the canary windows to create an incident report for a schema and latency regression that first appears during the 25% stage.

In [ ]:
incident_windows = decision_df[(decision_df["stage"].isin(["25%", "50%"])) & (decision_df["decision"] == "rollback")].copy()
incident_report = {
    "incident_id": "schema-regression-2026-05-18",
    "detected_at_minute": int(incident_windows["minute"].min()),
    "stage": incident_windows.iloc[0]["stage"],
    "primary_signal": "schema_pass_rate drop with latency increase",
    "supporting_signals": {
        "worst_schema_pass_rate": float(incident_windows["schema_pass_rate"].min()),
        "worst_p95_latency_ms": float(incident_windows["p95_latency_ms"].max()),
        "worst_error_rate": float(incident_windows["error_rate"].max()),
    },
    "actions": [
        "hold promotion at 25%",
        "sample traces for affected route",
        "execute rollback_plan.json",
        "run structured output spot checks after rollback",
    ],
    "status": "mitigated",
}
write_json(ARTIFACT_DIR / "incident_report.json", incident_report)
print(json.dumps(incident_report, indent=2))

## Step 11: Compute postmortem metrics and missed-gate analysis

After mitigation, review whether the gate policy caught the issue early enough and how much traffic saw the regression before rollback.

In [ ]:
rollback_minutes = decision_df[decision_df["decision"] == "rollback"]["minute"]
first_rollback_minute = int(rollback_minutes.min())
affected_windows = decision_df[decision_df["minute"] <= first_rollback_minute]
blast_radius_requests = int(affected_windows["candidate_requests"].sum())
avoidable_requests = int(decision_df[(decision_df["stage"] == "25%") & (decision_df["minute"] > first_rollback_minute)]["candidate_requests"].sum())

postmortem_rows = [
    {"metric": "first_rollback_minute", "value": first_rollback_minute},
    {"metric": "blast_radius_requests", "value": blast_radius_requests},
    {"metric": "avoidable_requests_after_detection", "value": avoidable_requests},
    {"metric": "blocked_suites", "value": int((comparison_df["status"] == "blocker").sum())},
    {"metric": "hold_windows", "value": int((decision_df["decision"] == "hold").sum())},
]
postmortem_df = pd.DataFrame(postmortem_rows)
postmortem_df.to_csv(ARTIFACT_DIR / "postmortem_metrics.csv", index=False)
print(postmortem_df.to_markdown(index=False))

## Step 12: Export a release packet for reviewers

A complete rollout review packet should contain the release comparison, canary results, rollback plan, incident playbooks, and postmortem metrics in one place.

In [ ]:
release_packet = {
    "module": "systems-05",
    "notebook": 19,
    "candidate_release": "candidate_2026_05_18",
    "baseline_release": "stable_2026_05_01",
    "comparison_artifact": "release_comparison.csv",
    "canary_artifact": "canary_windows.csv",
    "stage_decisions_artifact": "stage_decisions.csv",
    "rollback_plan_artifact": "rollback_plan.json",
    "incident_report_artifact": "incident_report.json",
    "playbook_index_artifact": "playbook_index.csv",
    "postmortem_artifact": "postmortem_metrics.csv",
}
write_json(ARTIFACT_DIR / "release_packet.json", release_packet)

summary_md = "\n".join([
    "# Module 5 Rollout Review",
    "",
    "## Suite comparison",
    comparison_df.to_markdown(index=False),
    "",
    "## Stage decisions",
    stage_decisions.to_markdown(index=False),
    "",
    "## Postmortem metrics",
    postmortem_df.to_markdown(index=False),
])
(ARTIFACT_DIR / "release_packet.md").write_text(summary_md, encoding="utf-8")
print(summary_md)

## Recap

You now have an eval-driven rollout workflow: compare releases, stage traffic, detect regressions, decide whether to hold or roll back, and execute a prepared incident playbook. This closes the loop from `evals/` into real operational governance.